# Missing Value Handling Strategy - TEDS-D
Analyze and handle missing values appropriately for statistical analysis and ML

## Load Data and Analyze Missing Patterns
Examine extent and patterns of missing data

In [2]:
import pandas as pd

df = pd.read_csv("../1_datasets/processed/tedsd_puf_2023_cleaned.csv")

## Calculate Missing Value Statistics
Get percentage of missing values for each column

In [3]:
missing_stats = pd.DataFrame(
    {
        "column": df.columns,
        "missing_count": df.isnull().sum(),
        "missing_percent": (df.isnull().sum() / len(df) * 100).round(2),
        "non_missing_count": df.notnull().sum(),
    }
).sort_values("missing_percent", ascending=False)
for col in [
    "patient_id",
    "service_type_admit",
    "primary_substance_admit",
    "age_group",
    "sex",
    "discharge_reason",
    "length_of_stay",
]:
    if col in df.columns:
        missing = df[col].isnull().sum()
        pct = missing / len(df) * 100
        print(f"{col:35} {missing:>8} ({pct:>5.2f}%)")

missing_stats

,column,missing_count,missing_percent,non_missing_count
DISYR,DISYR,0,0.0,1474025
CASEID,CASEID,0,0.0,1474025
STFIPS,STFIPS,0,0.0,1474025
EDUC,EDUC,0,0.0,1474025
MARSTAT,MARSTAT,0,0.0,1474025
...,...,...,...,...
DIVISION,DIVISION,0,0.0,1474025
REGION,REGION,0,0.0,1474025
IDU,IDU,0,0.0,1474025
ALCDRUG,ALCDRUG,0,0.0,1474025


## Identify Critical Variables
Define which variables are essential (cannot be missing for analysis)

In [4]:
df.columns.tolist()

['DISYR',
 'CASEID',
 'STFIPS',
 'EDUC',
 'MARSTAT',
 'SERVICES',
 'DETCRIM',
 'LOS',
 'PSOURCE',
 'NOPRIOR',
 'ARRESTS',
 'EMPLOY',
 'METHUSE',
 'PSYPROB',
 'PREG',
 'SEX',
 'VET',
 'LIVARAG',
 'DAYWAIT',
 'SERVICES_D',
 'REASON',
 'EMPLOY_D',
 'LIVARAG_D',
 'ARRESTS_D',
 'DSMCRIT',
 'AGE',
 'RACE',
 'ETHNIC',
 'DETNLF',
 'DETNLF_D',
 'PRIMINC',
 'SUB1',
 'SUB2',
 'SUB3',
 'SUB1_D',
 'SUB2_D',
 'SUB3_D',
 'ROUTE1',
 'ROUTE2',
 'ROUTE3',
 'FREQ1',
 'FREQ2',
 'FREQ3',
 'FREQ1_D',
 'FREQ2_D',
 'FREQ3_D',
 'FRSTUSE1',
 'FRSTUSE2',
 'FRSTUSE3',
 'HLTHINS',
 'PRIMPAY',
 'FREQ_ATND_SELF_HELP',
 'FREQ_ATND_SELF_HELP_D',
 'ALCFLG',
 'COKEFLG',
 'MARFLG',
 'HERFLG',
 'METHFLG',
 'OPSYNFLG',
 'PCPFLG',
 'HALLFLG',
 'MTHAMFLG',
 'AMPHFLG',
 'STIMFLG',
 'BENZFLG',
 'TRNQFLG',
 'BARBFLG',
 'SEDHPFLG',
 'INHFLG',
 'OTCFLG',
 'OTHERFLG',
 'DIVISION',
 'REGION',
 'IDU',
 'ALCDRUG',
 'CBSA2020']

In [5]:
critical_vars = [
    "CASEID",
    "SERVICES_D",
]

## Create Analysis-Ready Dataset (Minimal Removal)
Remove only rows missing critical variables for analysis

In [6]:
df_analysis = df.dropna(subset=critical_vars)

rows_removed = len(df) - len(df_analysis)
removal_percent = round(rows_removed / len(df) * 100, 2)

removal_summary = {
    "original_rows": len(df),
    "rows_after_removal": len(df_analysis),
    "rows_removed": rows_removed,
    "percent_removed": removal_percent,
}
df_analysis.to_csv("../1_datasets/processed/teds_d_analysis_ready.csv", index=False)

In [7]:
print(removal_summary)

{'original_rows': 1474025, 'rows_after_removal': 1474025, 'rows_removed': 0, 'percent_removed': 0.0}


## Machine Learning Preparation (For Later Phase)
For ML models, we'll need imputation rather than deletion

In [9]:
df_ml = df.copy()

numeric_cols = [
    "years_using",
    "number_of_substances_admit",
    "number_of_substances_discharge",
]
for col in numeric_cols:
    if col in df_ml.columns:
        df_ml[col] = df_ml[col].fillna(df_ml[col].median())

categorical_cols = [
    "wait_time_days",
    "prior_treatments",
    "employment_admit",
    "employment_discharge",
    "education_level",
    "living_arrangement_admit",
    "living_arrangement_discharge",
    "income_source",
    "length_of_stay",
    "discharge_reason",
]
for col in categorical_cols:
    if col in df_ml.columns:
        mode_val = df_ml[col].mode()
        if len(mode_val) > 0:
            df_ml[col] = df_ml[col].fillna(mode_val[0])

binary_cols = [
    col
    for col in df_ml.columns
    if col.startswith("is_")
    or col.startswith("has_")
    or col
    in [
        "completed_treatment",
        "dropped_out",
        "terminated",
        "transferred",
        "short_stay",
        "long_stay",
        "employment_improved",
        "housing_improved",
        "arrests_reduced",
    ]
]
for col in binary_cols:
    if col in df_ml.columns:
        df_ml[col] = df_ml[col].fillna(0)

remaining_missing = df_ml.isnull().sum()
cols_with_missing = remaining_missing[remaining_missing > 0]

if len(cols_with_missing) > 0:
    print(cols_with_missing)

    for col in cols_with_missing.index:
        if df_ml[col].dtype in ["object", "category"]:
            mode_val = df_ml[col].mode()
            fill_val = mode_val[0] if len(mode_val) > 0 else "Unknown"
            df_ml[col] = df_ml[col].fillna(fill_val)
        else:
            median_val = df_ml[col].median()
            fill_val = median_val if pd.notna(median_val) else 0
            df_ml[col] = df_ml[col].fillna(fill_val)

final_missing = df_ml.isnull().sum().sum()

df_ml.to_csv("../1_datasets/processed/teds_d_ml_ready.csv", index=False)

In [10]:
print(final_missing)

0
